In [1]:
from docx import Document

doc = Document("Shipping Email Segregation.docx")

text = "\n".join([p.text for p in doc.paragraphs])

print(text[:2000])

Shipping Email Segregation & Data Extraction Requirement
Objective
Create a system that automatically reads incoming shipping emails, categorises them, and extracts key commercial information for further processing and matching.
Note: The solution must NOT rely on third-party Large Language Model (LLM) APIs such as OpenAI ChatGPT, Anthropic Claude, or similar services. All processing should be performed using alternative AI/ML techniques, rule-based methods, or self-hosted models.

Email Categories
The system should automatically identify and segregate emails into the following categories:
1. Tonnage (Open Vessels)
Emails containing vessel availability information.
2. Cargo VC (Voyage Charter Cargo)
Emails containing cargo requirements for a specific voyage.
3. Cargo TC (Time Charter Cargo)
Emails containing time charter requirements.

Data to be Extracted
Tonnage Emails
Extract the following information:
Vessel Name
Account Name
Open Port
Open Date
Vessel Type
Vessel Size

Cargo VC Em

In [2]:
import re
import pandas as pd

pattern = r'MV\s+([A-Z\s]+?)\s+DWT\s+([\d,]+)\s+OPEN\s+(.+?)\s+O/A\s+(.+?)\n'

matches = re.findall(pattern, text, re.I)

rows = []

for m in matches:
    rows.append({
        "Vessel Name": m[0].strip(),
        "Vessel Size": m[1],
        "Open Port": m[2].strip(),
        "Open Date": m[3].strip(),
        "Category": "TONNAGE"
    })

df = pd.DataFrame(rows)

print(df.head())

df.to_csv("tonnage_data.csv", index=False)

    Vessel Name Vessel Size           Open Port      Open Date Category
0  SHENG AN HAI       56564       XIAMEN, china  2ND JUNE 2026  TONNAGE
1  FENG HUI HAI       63260    GUANGZHOU, CHINA  6TH JUNE 2026  TONNAGE
2  YUANPING SEA       55646         MANILA, PHI  3RD JUNE 2026  TONNAGE
3  SHENG DE HAI       56721  SAMALAJU, MALAYSIA  3RD JUNE 2026  TONNAGE
4   BI JIA SHAN       56623    GWADAR, PAKISTAN  2ND JUNE 2026  TONNAGE


In [3]:
import re

vessels = re.findall(r'MV[\./]?\s*([A-Z\s]+)', text, re.I)

print("Total vessels found:", len(vessels))

Total vessels found: 20


In [4]:
vc_start = text.find("VC CARGO")
tc_start = text.find("TC CARGO")

vc_text = text[vc_start:tc_start]

print(vc_text)

VC CARGO

TELiX MSG: 09A76-00 22/05/26 13:32 +03:00

﻿22 MAY 2026 

ATTN CHARTERING DESK !

DEAR SIR

GOOD DAY

PLEASE OFFER FIRM FOR FOLL FULY FIRM CARGO

15,000 - 20,000 MTS 10PCT MOLOCHOPT
LOAD PORT : KOH SI CHANG , THAILAND
DISCHARGE PORT: KANDLA + CHENNAI
LOAD RATE: 1,000 MTS PWWD SSHEX
DISCHARGE RATE: 1500 MTS PWWD SSHEX
LAYCAN: MID JULY 2026
COM : 3.75 PCT TTL

—-------------------------------------------------------------------------------------------------------------------------

MCD LIDOMAR
Att. Chartering Desk
/Catalin
Good day

Jeddah  / Bilbao 

20 000  mt HRC  max 28,5 mt

FIOS

4000 mt fhinc / CQD disch

25 June - 5 July try later

3,75% here

-- 
Best   regards/Cristescu
As brokers only ,
Phone: (+) 40-751-991111
EMAIL lidomar@lidomar.ro
re:7647terdhf
Skype  cccatalin
Please add our email to your regular distribution list.
In case the content of our messages are not of your interest please
kindly advise in return  in order to remove you from the distribution list.

+++

In [5]:
tc_text = text[tc_start:]

print(tc_text)

TC CARGO

TELiX MSG: 0C2V5-00 22/05/26 10:45 +03:00

- / KP

GOOD DAY,

PLEASED TO HEAR.

CHINA / NOPAC
ACC DAI AN OCEAN SHIPPING COMPANY LIMITED
DELIVERY TM VANCOUVER
LC 10-17 JUNE
SMX-UMX, PREF UMX
1 TCT WITH GRAINS
REDELIVERY CHITTAGONG
3.75 ADDCOM PUS
—------------------------------------------------------------------------------------------------------------
SEASIA
ACC DAI AN OCEAN SHIPPING COMPANY LIMITED
SMX-UMX MAX 20 YRS.
DELY TO MAKE SANGATTA (NEAR TO TJ BARA), E KALI OF INDONESIA.
29-2ND JUN
1 TCT WITH CLINKER TO BDESH.
DURATION ABT 30 DAYS WOG.
3.75PCT ADDOM PUS
TRY PERIOD
PLS ADV THE OUTREACH
—------------------------------------------------------------------------------------------------------------
WORLDWIDE
ACC DAI AN OCEAN SHIPPING COMPANY LIMITED
SUPRA/ULTRA DELY WW
FULL MAY
1-3 YEARS TRY SHORT PERIOD
FLAT OR INDEX BOTH WORKABLE
3.75 ADDCOM PUS
—----------------------------------------------------------------------------------------------------------------------------

In [6]:
import re

load_ports = re.findall(r'LOAD PORT\s*:\s*(.*)', vc_text, re.I)
discharge_ports = re.findall(r'DISCHARGE PORT\s*:\s*(.*)', vc_text, re.I)
laycans = re.findall(r'LAYCAN\s*:\s*(.*)', vc_text, re.I)

print(load_ports)
print(discharge_ports)
print(laycans)

['KOH SI CHANG , THAILAND']
['KANDLA + CHENNAI']
['MID JULY 2026', '16-20 July']


In [7]:
delivery = re.findall(r'DELIVERY\s+(.*)', tc_text, re.I)
redelivery = re.findall(r'REDELIVERY\s+(.*)', tc_text, re.I)
duration = re.findall(r'DURATION\s+(.*)', tc_text, re.I)

print(delivery)
print(redelivery)
print(duration)

['TM VANCOUVER', 'CHITTAGONG']
['CHITTAGONG']
['ABT 30 DAYS WOG.']


In [8]:
import pandas as pd
import re

vc_rows = []

# Record 1
m = re.search(
    r'(\d[\d,\-\s]+MTS.*)\nLOAD PORT\s*:\s*(.*?)\nDISCHARGE PORT:\s*(.*?)\n.*?LAYCAN:\s*(.*?)\n',
    vc_text,
    re.I | re.S
)

if m:
    vc_rows.append({
        "Cargo Name": m.group(1).strip(),
        "Loading Port": m.group(2).strip(),
        "Discharge Port": m.group(3).strip(),
        "Laycan": m.group(4).strip(),
        "Category": "CARGO_VC"
    })

# Record 2 (Jeddah / Bilbao)
m = re.search(
    r'Jeddah\s*/\s*Bilbao.*?\n\n(.*?)\n.*?\n\n(25 June - 5 July.*?)\n',
    vc_text,
    re.I | re.S
)

if m:
    vc_rows.append({
        "Cargo Name": m.group(1).strip(),
        "Loading Port": "Jeddah",
        "Discharge Port": "Bilbao",
        "Laycan": m.group(2).strip(),
        "Category": "CARGO_VC"
    })

# Record 3
m = re.search(
    r'20-30,000 mts iron slag in bulk\s*LP:(.*?)\s*DP:(.*?)\s*10000/12000\s*(.*?)\s*3.75',
    vc_text,
    re.I | re.S
)

if m:
    vc_rows.append({
        "Cargo Name": "Iron Slag",
        "Loading Port": m.group(1).strip(),
        "Discharge Port": m.group(2).strip(),
        "Laycan": m.group(3).strip(),
        "Category": "CARGO_VC"
    })

# Record 4
m = re.search(
    r'Cargo:30,000 mts of Urea in bulk\s*POL:(.*?)\s*POD:(.*?)\s*5000/5000\s*LAYCAN:\s*(.*?)\n',
    vc_text,
    re.I | re.S
)

if m:
    vc_rows.append({
        "Cargo Name": "Urea",
        "Loading Port": m.group(1).strip(),
        "Discharge Port": m.group(2).strip(),
        "Laycan": m.group(3).strip(),
        "Category": "CARGO_VC"
    })

vc_df = pd.DataFrame(vc_rows)

print(vc_df)

vc_df.to_csv("cargo_vc.csv", index=False)

                            Cargo Name             Loading Port  \
0  15,000 - 20,000 MTS 10PCT MOLOCHOPT  KOH SI CHANG , THAILAND   
1          20 000  mt HRC  max 28,5 mt                   Jeddah   
2                            Iron Slag                  Bushehr   
3                                 Urea                      BIk   

         Discharge Port                      Laycan  Category  
0      KANDLA + CHENNAI               MID JULY 2026  CARGO_VC  
1                Bilbao  25 June - 5 July try later  CARGO_VC  
2                  Doha                  25-30 july  CARGO_VC  
3  Iskenderun or Durban                  16-20 July  CARGO_VC  


In [9]:
tc_start = text.find("TC CARGO")

tc_text = text[tc_start:]

print(tc_text)

TC CARGO

TELiX MSG: 0C2V5-00 22/05/26 10:45 +03:00

- / KP

GOOD DAY,

PLEASED TO HEAR.

CHINA / NOPAC
ACC DAI AN OCEAN SHIPPING COMPANY LIMITED
DELIVERY TM VANCOUVER
LC 10-17 JUNE
SMX-UMX, PREF UMX
1 TCT WITH GRAINS
REDELIVERY CHITTAGONG
3.75 ADDCOM PUS
—------------------------------------------------------------------------------------------------------------
SEASIA
ACC DAI AN OCEAN SHIPPING COMPANY LIMITED
SMX-UMX MAX 20 YRS.
DELY TO MAKE SANGATTA (NEAR TO TJ BARA), E KALI OF INDONESIA.
29-2ND JUN
1 TCT WITH CLINKER TO BDESH.
DURATION ABT 30 DAYS WOG.
3.75PCT ADDOM PUS
TRY PERIOD
PLS ADV THE OUTREACH
—------------------------------------------------------------------------------------------------------------
WORLDWIDE
ACC DAI AN OCEAN SHIPPING COMPANY LIMITED
SUPRA/ULTRA DELY WW
FULL MAY
1-3 YEARS TRY SHORT PERIOD
FLAT OR INDEX BOTH WORKABLE
3.75 ADDCOM PUS
—----------------------------------------------------------------------------------------------------------------------------

In [10]:
df = pd.DataFrame(rows)

In [11]:
tonnage_df = df

In [12]:
tc_start = text.find("TC CARGO")

tc_text = text[tc_start:]

print(tc_text)

TC CARGO

TELiX MSG: 0C2V5-00 22/05/26 10:45 +03:00

- / KP

GOOD DAY,

PLEASED TO HEAR.

CHINA / NOPAC
ACC DAI AN OCEAN SHIPPING COMPANY LIMITED
DELIVERY TM VANCOUVER
LC 10-17 JUNE
SMX-UMX, PREF UMX
1 TCT WITH GRAINS
REDELIVERY CHITTAGONG
3.75 ADDCOM PUS
—------------------------------------------------------------------------------------------------------------
SEASIA
ACC DAI AN OCEAN SHIPPING COMPANY LIMITED
SMX-UMX MAX 20 YRS.
DELY TO MAKE SANGATTA (NEAR TO TJ BARA), E KALI OF INDONESIA.
29-2ND JUN
1 TCT WITH CLINKER TO BDESH.
DURATION ABT 30 DAYS WOG.
3.75PCT ADDOM PUS
TRY PERIOD
PLS ADV THE OUTREACH
—------------------------------------------------------------------------------------------------------------
WORLDWIDE
ACC DAI AN OCEAN SHIPPING COMPANY LIMITED
SUPRA/ULTRA DELY WW
FULL MAY
1-3 YEARS TRY SHORT PERIOD
FLAT OR INDEX BOTH WORKABLE
3.75 ADDCOM PUS
—----------------------------------------------------------------------------------------------------------------------------

In [13]:
tonnage_df.to_csv("tonnage.csv", index=False)

vc_df.to_csv("cargo_vc.csv", index=False)

In [14]:
import os

print(os.listdir())
print(tc_text)

['.ipynb_checkpoints', 'cargo_tc.csv', 'cargo_vc.csv', 'Shipping Email Segregation.docx', 'shipping_dataset.csv', 'shipping_model.pkl', 'tonnage.csv', 'tonnage_data.csv', 'Untitled.ipynb', 'Untitled1.ipynb', 'vectorizer.pkl']
TC CARGO

TELiX MSG: 0C2V5-00 22/05/26 10:45 +03:00

- / KP

GOOD DAY,

PLEASED TO HEAR.

CHINA / NOPAC
ACC DAI AN OCEAN SHIPPING COMPANY LIMITED
DELIVERY TM VANCOUVER
LC 10-17 JUNE
SMX-UMX, PREF UMX
1 TCT WITH GRAINS
REDELIVERY CHITTAGONG
3.75 ADDCOM PUS
—------------------------------------------------------------------------------------------------------------
SEASIA
ACC DAI AN OCEAN SHIPPING COMPANY LIMITED
SMX-UMX MAX 20 YRS.
DELY TO MAKE SANGATTA (NEAR TO TJ BARA), E KALI OF INDONESIA.
29-2ND JUN
1 TCT WITH CLINKER TO BDESH.
DURATION ABT 30 DAYS WOG.
3.75PCT ADDOM PUS
TRY PERIOD
PLS ADV THE OUTREACH
—------------------------------------------------------------------------------------------------------------
WORLDWIDE
ACC DAI AN OCEAN SHIPPING COMPANY LIMITED

In [15]:
import pandas as pd

tc_rows = [
    {
        "Account Name": "DAI AN OCEAN SHIPPING COMPANY LIMITED",
        "Cargo Name": "Grains",
        "Delivery Port": "Vancouver",
        "Redelivery Port": "Chittagong",
        "Duration": "1 TCT",
        "Laycan": "10-17 June",
        "Category": "CARGO_TC"
    },
    {
        "Account Name": "DAI AN OCEAN SHIPPING COMPANY LIMITED",
        "Cargo Name": "Clinker",
        "Delivery Port": "Sangatta, Indonesia",
        "Redelivery Port": "Bangladesh",
        "Duration": "30 Days",
        "Laycan": "29-2 June",
        "Category": "CARGO_TC"
    },
    {
        "Account Name": "SeaSchiffe",
        "Cargo Name": "Steels/Gens/Lawfuls",
        "Delivery Port": "ECI",
        "Redelivery Port": "Med via GOA transit",
        "Duration": "35-40 days",
        "Laycan": "15-18 July",
        "Category": "CARGO_TC"
    },
    {
        "Account Name": "SeaSchiffe",
        "Cargo Name": "Steels/Gens/Lawfuls",
        "Delivery Port": "ECI",
        "Redelivery Port": "ARAG via COGH transit",
        "Duration": "50-55 days",
        "Laycan": "21-23 July",
        "Category": "CARGO_TC"
    }
]

tc_df = pd.DataFrame(tc_rows)

print(tc_df)

tc_df.to_csv("cargo_tc.csv", index=False)

                            Account Name           Cargo Name  \
0  DAI AN OCEAN SHIPPING COMPANY LIMITED               Grains   
1  DAI AN OCEAN SHIPPING COMPANY LIMITED              Clinker   
2                             SeaSchiffe  Steels/Gens/Lawfuls   
3                             SeaSchiffe  Steels/Gens/Lawfuls   

         Delivery Port        Redelivery Port    Duration      Laycan  \
0            Vancouver             Chittagong       1 TCT  10-17 June   
1  Sangatta, Indonesia             Bangladesh     30 Days   29-2 June   
2                  ECI    Med via GOA transit  35-40 days  15-18 July   
3                  ECI  ARAG via COGH transit  50-55 days  21-23 July   

   Category  
0  CARGO_TC  
1  CARGO_TC  
2  CARGO_TC  
3  CARGO_TC  


In [16]:
final_df = pd.concat(
    [tonnage_df, vc_df, tc_df],
    ignore_index=True
)

final_df.to_csv("shipping_dataset.csv", index=False)

print("Total Records:", len(final_df))
print(final_df.head())

Total Records: 15
    Vessel Name Vessel Size           Open Port      Open Date Category  \
0  SHENG AN HAI       56564       XIAMEN, china  2ND JUNE 2026  TONNAGE   
1  FENG HUI HAI       63260    GUANGZHOU, CHINA  6TH JUNE 2026  TONNAGE   
2  YUANPING SEA       55646         MANILA, PHI  3RD JUNE 2026  TONNAGE   
3  SHENG DE HAI       56721  SAMALAJU, MALAYSIA  3RD JUNE 2026  TONNAGE   
4   BI JIA SHAN       56623    GWADAR, PAKISTAN  2ND JUNE 2026  TONNAGE   

  Cargo Name Loading Port Discharge Port Laycan Account Name Delivery Port  \
0        NaN          NaN            NaN    NaN          NaN           NaN   
1        NaN          NaN            NaN    NaN          NaN           NaN   
2        NaN          NaN            NaN    NaN          NaN           NaN   
3        NaN          NaN            NaN    NaN          NaN           NaN   
4        NaN          NaN            NaN    NaN          NaN           NaN   

  Redelivery Port Duration  
0             NaN      NaN  
1   

In [17]:
print("Tonnage:", len(tonnage_df))
print("VC:", len(vc_df))
print("TC:", len(tc_df))

Tonnage: 7
VC: 4
TC: 4


In [18]:
print(final_df.tail(10))

     Vessel Name Vessel Size                Open Port      Open Date  \
5   yuanning sea       55580              SOHAR, OMAN  30TH may 2026   
6     COS ORCHID       55550  DAR ES SALAAM, TANZANIA  1ST JUNE 2026   
7            NaN         NaN                      NaN            NaN   
8            NaN         NaN                      NaN            NaN   
9            NaN         NaN                      NaN            NaN   
10           NaN         NaN                      NaN            NaN   
11           NaN         NaN                      NaN            NaN   
12           NaN         NaN                      NaN            NaN   
13           NaN         NaN                      NaN            NaN   
14           NaN         NaN                      NaN            NaN   

    Category                           Cargo Name             Loading Port  \
5    TONNAGE                                  NaN                      NaN   
6    TONNAGE                                  NaN  

In [19]:
final_df.to_csv("shipping_dataset.csv", index=False)

In [20]:
import pandas as pd

df = pd.read_csv("shipping_dataset.csv")

print(df.head())
print(df.columns)

    Vessel Name  Vessel Size           Open Port      Open Date Category  \
0  SHENG AN HAI      56564.0       XIAMEN, china  2ND JUNE 2026  TONNAGE   
1  FENG HUI HAI      63260.0    GUANGZHOU, CHINA  6TH JUNE 2026  TONNAGE   
2  YUANPING SEA      55646.0         MANILA, PHI  3RD JUNE 2026  TONNAGE   
3  SHENG DE HAI      56721.0  SAMALAJU, MALAYSIA  3RD JUNE 2026  TONNAGE   
4   BI JIA SHAN      56623.0    GWADAR, PAKISTAN  2ND JUNE 2026  TONNAGE   

  Cargo Name Loading Port Discharge Port Laycan Account Name Delivery Port  \
0        NaN          NaN            NaN    NaN          NaN           NaN   
1        NaN          NaN            NaN    NaN          NaN           NaN   
2        NaN          NaN            NaN    NaN          NaN           NaN   
3        NaN          NaN            NaN    NaN          NaN           NaN   
4        NaN          NaN            NaN    NaN          NaN           NaN   

  Redelivery Port Duration  
0             NaN      NaN  
1             Na

In [21]:
df = df.fillna("")

In [22]:
df["email_text"] = (
    df["Vessel Name"].astype(str) + " " +
    df["Cargo Name"].astype(str) + " " +
    df["Open Port"].astype(str) + " " +
    df["Loading Port"].astype(str) + " " +
    df["Delivery Port"].astype(str)
)

print(df[["email_text", "Category"]].head())

                           email_text Category
0       SHENG AN HAI  XIAMEN, china    TONNAGE
1    FENG HUI HAI  GUANGZHOU, CHINA    TONNAGE
2         YUANPING SEA  MANILA, PHI    TONNAGE
3  SHENG DE HAI  SAMALAJU, MALAYSIA    TONNAGE
4     BI JIA SHAN  GWADAR, PAKISTAN    TONNAGE


In [23]:
print(df["Category"].value_counts())

Category
TONNAGE     7
CARGO_VC    4
CARGO_TC    4
Name: count, dtype: int64


In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

X = df["email_text"]
y = df["Category"]

vectorizer = TfidfVectorizer()

X_vec = vectorizer.fit_transform(X)

print(X_vec.shape)

(15, 58)


In [25]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()

model.fit(X_vec, y)

print("Model trained successfully")

Model trained successfully


In [26]:
sample = ["LOAD PORT KOH SI CHANG DISCHARGE PORT KANDLA"]

sample_vec = vectorizer.transform(sample)

prediction = model.predict(sample_vec)

print(prediction)

['CARGO_VC']


In [27]:
import joblib

joblib.dump(model, "shipping_model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")

print("Model Saved")

Model Saved


In [28]:
import os

print(os.listdir())

['.ipynb_checkpoints', 'cargo_tc.csv', 'cargo_vc.csv', 'Shipping Email Segregation.docx', 'shipping_dataset.csv', 'shipping_model.pkl', 'tonnage.csv', 'tonnage_data.csv', 'Untitled.ipynb', 'Untitled1.ipynb', 'vectorizer.pkl']


In [29]:
import joblib

model = joblib.load("shipping_model.pkl")
vectorizer = joblib.load("vectorizer.pkl")

print("Model Loaded Successfully")

Model Loaded Successfully


In [30]:
sample = ["MV NEW STAR DWT 50000 OPEN SINGAPORE"]

sample_vec = vectorizer.transform(sample)

prediction = model.predict(sample_vec)

print(prediction)

['TONNAGE']


In [31]:
sample = ["LOAD PORT KOH SI CHANG DISCHARGE PORT KANDLA"]

sample_vec = vectorizer.transform(sample)

prediction = model.predict(sample_vec)

print(prediction)

['CARGO_VC']


In [32]:
sample = ["DELIVERY VANCOUVER REDELIVERY CHITTAGONG"]

sample_vec = vectorizer.transform(sample)

prediction = model.predict(sample_vec)

print(prediction)

['CARGO_TC']


In [33]:
sample = ["LOAD PORT KOH SI CHANG DISCHARGE PORT KANDLA"]

sample_vec = vectorizer.transform(sample)

prediction = model.predict(sample_vec)

print("Predicted Category:", prediction[0])

Predicted Category: CARGO_VC


In [34]:
sample = ["DELIVERY VANCOUVER REDELIVERY CHITTAGONG"]

sample_vec = vectorizer.transform(sample)

prediction = model.predict(sample_vec)

print("Predicted Category:", prediction[0])

Predicted Category: CARGO_TC
